In [0]:
import functools
import os, pathlib

from typing import Collection, Dict, List

import IPython
import pyspark.sql.functions as F
import yaml
from pyspark.sql.column import Column


def list_directories(path, max_depth=None) -> List:
    """
    List all directories recursively from the given path, with optional filtering and depth control.
    Args:
        path: The base directory path to start listing from.
        max_depth: An integer specifying the maximum depth to traverse (default is None, traverse all depths).
    Returns:
        A list of directories based on the given parameters.
    """
    result = []

    for root, dirs, files in os.walk(path):
        # Calculate the relative root and depth from the base path
        relative_root = os.path.relpath(root, path)
        depth = relative_root.count(os.path.sep) + (1 if relative_root != "." else 0)

        if max_depth is not None and depth >= max_depth:
            break

        for d in dirs:
            result.append(d) if relative_root == "." else result.append(os.path.join(relative_root, d))

    return result


def list_files(path, filter_name="README", max_depth=None, full_path=False, stem_only=True) -> List:
    """
    List all files recursively from the given path, with optional filtering and depth control.
    Args:
        path: The base directory path to start listing from.
        filter_name: A string to filter out files and directories starting with this value (default is "README").
        max_depth: An integer specifying the maximum depth to traverse (default is None, traverse all depths).
        full_path: Indicates whether to provide the relative path or just the file name
    Returns:
        A list of files based on the given parameters.
    """
    result = []

    for root, dirs, files in os.walk(path):
        # Calculate the relative root and depth from the base path
        relative_root = os.path.relpath(root, path)
        depth = relative_root.count(os.path.sep) + (1 if relative_root != "." else 0)

        if max_depth is not None and depth >= max_depth:
            break

        for f in files:
            f = pathlib.Path(f)
            file_name = f.stem if stem_only else f.name
            if not file_name.startswith(filter_name):
                result.append(os.path.join(relative_root, file_name)) if full_path else result.append(f.stem)

    return result


def get_object_config(path) -> Dict:
    """
    Handles loading a YAML configuration file and returns a dictionary

    Args:
        path: str | Path
            Configuraton file path
    Returns:
        Dictionary object with the contents of the config file
    """
    # Ensure the path is a pathlib.Path object
    if not isinstance(path, pathlib.Path):
        path = pathlib.Path(path)

    # add suffix to name, if not already present
    path = path.with_suffix(".yml") if path.suffix != ".yml" else path

    # Check for YAML file
    if path.exists():
        with open(path, "r") as file:
            return yaml.safe_load(file)
    else:
        raise FileNotFoundError(f"YAML file not found for the given path: {path}")


def sap_space_or_null_to_empty(col: Column) -> Column:
    """
    A function to check if a given Spark DataFrame column is with one single space or null.

    Args:
        col (pyspark.sql.Column):
            A column of Spark DataFrame to check for one single space or null values.

    Returns:
        pyspark.sql.Column:
            A new column with empty strings for null or one single space values in the input column.
    """
    return F.when((col.isNotNull()) & (col != F.lit(" ")), col).otherwise(F.lit(""))


def sap_space_or_null_to_empty_with_fallback(col: Column, fallback_col: Column) -> Column:
    """
    A function to check if a given Spark DataFrame column is with one single space or null, and return
    a fallback value if the input column is with one single space or null.

    Args:
        col (pyspark.sql.Column):
            A column of Spark DataFrame to check for single space or null values.
        fallback_col (pyspark.sql.Column):
            A fallback column of Spark DataFrame to use if the input column is with single space or null.

    Returns:
        pyspark.sql.Column:
            A new column with values from the input column, fallback column, or an empty string.
    """

    return F.when((col.isNotNull()) & (col != F.lit(" ")), col).otherwise(
        F.when((fallback_col.isNotNull()) & (fallback_col != F.lit(" ")), fallback_col).otherwise(F.lit(""))
    )


def sap_blank_to_empty_with_fallback(col: Column, fallback_col: Column) -> Column:
    """
    A function to check if a given Spark DataFrame column is empty, with one single space or null, and return
    a fallback value if the input column is empty, with one single space or null.

    Args:
        col (pyspark.sql.Column):
            A column of Spark DataFrame to check for empty, with one single space or null values.
        fallback_col (pyspark.sql.Column):
            A fallback column of Spark DataFrame to use if the input column is empty, with one single space or null.

    Returns:
        pyspark.sql.Column:
            A new column with values from the input column, fallback column, or an empty string.
    """

    return F.when((col.isNotNull()) & (col != F.lit(" ")) & (col != F.lit("")), col).otherwise(
        F.when((fallback_col.isNotNull()) & (fallback_col != F.lit(" ")), fallback_col).otherwise(F.lit(""))
    )


def sap_default_date_replace(col: Column, return_on_true: str = None, return_on_false: str = None) -> Column:
    """
    A function to check if a given Spark DataFrame column is not equal to a default date value and return a value based on a condition.

    Args:
        col (pyspark.sql.Column):
            A column of Spark DataFrame to check for the default date value.
        return_on_true (str, optional):
            A string value to return when the input column is not equal to the default date value.
            If not provided, the input column's value will be returned.
        return_on_false (str, optional):
            A string value to return when the input column is equal to the default date value.
            If not provided, None will be returned.

    Returns:
        pyspark.sql.Column:
            A new column with values based on the input column and the provided return values.
    """
    if return_on_true and return_on_false:
        return F.when((col.isNotNull()) & (col != F.lit("0101-01-01")), F.lit(return_on_true)).otherwise(
            F.lit(return_on_false)
        )
    else:
        return F.when((col.isNotNull()) & (col != F.lit("0101-01-01")), col).otherwise(F.lit(None))


def sap_default_date_replace_with_fallback(col: Column, fallback_col: Column) -> Column:
    """
    A function to check if a given Spark DataFrame column is not equal to a default date value and return a fallback value.

    Args:
        col (pyspark.sql.Column):
            A column of Spark DataFrame to check for the default date value.
        rfallback_col (pyspark.sql.Column):
            A fallback column of Spark DataFrame to use if the input column is empty or "0101-01-01".

    Returns:
        pyspark.sql.Column:
            A new column with values based on the input column or the provided fallback column.
    """
    return F.when((col.isNotNull()) & (col != F.lit("0101-01-01")), col).otherwise(
        F.when(
            (fallback_col.isNotNull()) & (fallback_col != F.lit("0101-01-01")),
            fallback_col,
        ).otherwise(F.lit(None))
    )


def sap_string_sanitize(col: Column) -> Column:
    """
    A function to sanitize a given Spark DataFrame column by removing extra white spaces, unwanted tabs and line breakers, and fixing encoding issues.

    Args:
        col (pyspark.sql.Column):
            A column of Spark DataFrame to sanitize.

    Returns:
        pyspark.sql.Column:
            A new column with sanitized values based on the input column.
    """

    col_sanitized = F.regexp_replace(
        F.trim(col), "\s{2,}|\\n+", " "
    )  # Fix extra white spaces and replace line breakers (\n) by 1 white space
    col_sanitized = F.regexp_replace(
        col_sanitized, "\\t+|(\\xa0)+|^\\s+|\\s+$", ""
    )  # Remove unwanted tabs, white chars and left spaces at the begining or ending
    col_sanitized = F.decode(F.encode(col_sanitized, "ISO-8859-1"), "CP1252")  # Fix encoding

    return col_sanitized


def modify_dbfs_path(in_path: str) -> pathlib.Path:
    """handles if path has dbfs:/ and modifies to /dbfs/

    Args:
        in_path: str
    Returns:
        pathlib.Path with 'dbfs:/' changed to '/dbfs' if present
    """
    path = pathlib.Path(in_path)
    if path.parts[0] == "dbfs:":
        path = pathlib.Path("/dbfs", *path.parts[1:])
    return path


def file_exists(dst):
    """Checks if file exists in the destination location

    Args:
        dst (str): destination file path

    Returns:
        boolean : True or False

    """
    dbutils = IPython.get_ipython().user_ns["dbutils"]  # DBUtils(spark)
    try:
        dbutils.fs.ls(dst)
    except Exception:
        return False
    return True


def filter_dict(in_dict: dict, excludes: Collection = (None,)) -> dict:
    """Filters target values from a dictionary

    Args:
        in_dict (dict): Dictionary to filter values from
        excludes (Collection, optional): Values to filter from the dictionary. Defaults to (None,).

    Returns:
        dict: without keys containing filtered values

    Example:
       >>> filter_dict({'a': None, 'b': 1})
       ... {'b': 1}
    """
    return {k: v for k, v in in_dict.items() if v not in excludes}
